# 7. Analysis: trade studies, sizing, and requirement consistency

`longeron.analysis` projects executable models onto external solvers.
Three submodules, each behind its own extra, so the core package stays
dependency-light:

| Module | Solver | Install | Answers |
|---|---|---|---|
| `analysis.trades` | OR-Tools CP-SAT | `pip install "longeron[trades]"` | which discrete component mixes are feasible / optimal, and why not |
| `analysis.mdao` | OpenMDAO | `pip install "longeron[mdao]"` | continuous sizing, what-ifs, gradient-based optimization, external-tool binding |
| `analysis.smt` | Z3 | `pip install "longeron[smt]"` | is the requirement set consistent at all, which requirements conflict, exact feasibility bounds |

The interpreter remains the single source of semantics: every solver
result below is re-evaluated, or cross-checked, against the model
itself.

**You will learn how to:**

- score a discrete architecture catalog exactly through the
  interpreter and read one Pareto front per mission;
- see why the infeasible mixes die, in a census and in a constraint
  network;
- brush the design space in linked widgets, up to the
  mission-compromise dashboard;
- size a frozen winner with OpenMDAO, read the generated problem's N2
  structure, and swap in a declared higher-fidelity analysis;
- check requirement consistency, and extract conflict cores, with Z3.

**Prerequisites:** tutorials 3 and 6, plus the analysis extras:
`pip install "longeron[trades,mdao,smt,viz]"`. The widgets need
JupyterLab; on a static page they show placeholders. The final CAD cell
needs the `cad` extra and degrades to a note without it.

In [ ]:
import longeron
from longeron.analysis import mdao, smt, trades

## A multi-mission UAV catalog

`examples/uav_missions.sysml` models one component catalog: four
airframe families, tiered motors, props, batteries, mission equipment,
and a **structural material** choice. The catalog is evaluated against
**three mission contexts**, each a part definition specializing the
shared `MissionUAV` assembly with its own equipment, metrics, and
requirements:

* **`IsrUav`** -- loiter on station with a stabilized sensor (metric:
  `stationMinutes`);
* **`LogisticsUav`** -- fly a parcel out and return *empty*, the two legs
  costing different power (metric: `payloadRangeKgKm`);
* **`InterceptUav`** -- a one-way dash to catch a target crossing at 25
  m/s, first seen 3 km out (metric: `maxTargetSpeed`).

The airframes are genuinely different machines:

* a cheap rotor-only **box quad**;
* a **teardrop quad**: the same four-rotor lift inside a lathed
  low-drag shell, added to probe whether wings are necessary at all
  (its slim bay carries almost nothing);
* a **winged VTOL** that hovers on four props but cruises on its 2.6 m
  wing. The two large props sit on *wingtip nacelles*, where they
  rotate against the tip vortices (modeled as a 1.28x bonus on the
  wing's effective span efficiency, that is, less induced drag), with
  two smaller lift props atop the twin vertical stabilizers;
* a rail-launched **streamlined interceptor** (single pusher motor, no
  hover requirement, and almost no payload).

The airframes are fictional; everything bolted to them is a real
commercial part with nominal catalog figures: T-Motor Antigravity
MN4006 / SunnySky X4112S / T-Motor AT4120 motors, APC and T-Motor
props, three Tattu LiPo packs plus an 18650 li-ion pack (the chemistry
axis: more watt-hours per kilogram, a tenth of the discharge ceiling),
a RunCam nose camera, DJI and Gremsy gimbals, and drop bays.

All the physics lives in `calc def`s the interpreter evaluates
directly, so the model itself carries the analysis: momentum-theory
hover power from disk loading, a **wetted-area parasite-drag buildup**
(no airframe gets its CdA by fiat), parasite + induced-drag cruise
power, drag-limited dash speed, the lead-collision intercept triangle,
and **load-based structural sizing** (rotor arms and wing spars as
bending-sized tubes in aluminum or carbon). The calc defs are organized
into **discipline packages** (`Aerodynamics`, `Propulsion`,
`Structures`, `Performance`), and that organization is not cosmetic:
the MDAO bridge later groups the generated problem by exactly these
packages, so the N2 diagram tells the classic discipline story because
the model does.

In [ ]:
model = longeron.load("../examples/uav_missions.sysml")
missions = {
    "ISR": ("UavMissions::IsrUav", "stationMinutes"),
    "logistics": ("UavMissions::LogisticsUav", "payloadRangeKgKm"),
    "intercept": ("UavMissions::InterceptUav", "maxTargetSpeed"),
}
studies = {name: trades.TradeStudy(model, qname) for name, (qname, _) in missions.items()}
for name, study in studies.items():
    points = ", ".join(f"{p.name}[{len(p.variants)}]" for p in study.points.values())
    print(f"{name:9s} -> {points}")

### The catalog, in a traditional SysML structure diagram

Before any analysis, let the model introduce itself.
`UavMissions::Catalog` holds the seven variation points under trade.

What to look for in the diagram below: the variation tree (airframes,
motors, props, batteries, sensors, bays, materials) is exactly the set
of columns you will brush in every parallel-coordinates view further
down. (The diagram needs the vendored ipyelk; the cell degrades to a
note without it.)

In [ ]:
try:
    from longeron import diagrams

    display(diagrams.structure_diagram(model.find("UavMissions::Catalog"), show_attributes=False))
except ImportError:
    print("optional: pip install -e vendor/ipyelk enables the SysML diagrams -- skipping here")

### The requirements, as the model states them

`UavMissions::MissionRequirements` formalizes each tasking as a
requirement def: subject, `assume`, and `require` text straight from
the model. The numeric floors those constraints reference
(`minStationMinutes`, `minPayloadKg`, `minDeliveryRadiusKm`,
`targetSpeed`) are the same attributes the mission dashboard at the end
of this notebook reads its requirement-slider defaults from.

In [ ]:
try:
    from longeron import diagrams

    display(diagrams.structure_diagram(model.find("UavMissions::MissionRequirements")))
except ImportError:
    print("optional: pip install -e vendor/ipyelk enables the SysML diagrams -- skipping here")

### The honest solver choice at this scale

CP-SAT's fixed-point integer arithmetic covers linear-ish catalogs
(`examples/drone_catalog.sysml` still demos `enumerate`/`explain` on
it). But `sqrt`, `pow`, conditionals, and calc invocations are beyond
the mapper, and it says so instead of silently mis-encoding. With at most 864
candidates per mission there is nothing to prune anyway:
`all_architectures()` walks the whole Cartesian space through the
interpreter, exactly, in well under a second. Each infeasible mix
carries `violations`, the names of the constraints it breaks.

In [ ]:
try:
    studies["ISR"].enumerate()
except longeron.analysis.AnalysisError as err:
    print(f"CP-SAT mapper: {err}\n")

spaces = {name: study.all_architectures() for name, study in studies.items()}
for name, archs in spaces.items():
    feasible = sum(a.verified for a in archs)
    print(f"{name:9s} {feasible:3d} of {len(archs)} mixes feasible")

### Where the drag numbers come from

No airframe quotes its CdA by fiat: every `dragArea` is a **wetted-area
buildup** in the model itself. The buildup is `CdA = sum over surfaces
of Cf x S_wet x form factor`: flat-plate skin friction at a fixed
representative Cf = 0.0055, Hoerner-style form factors (slender bodies
~1.1-1.25 by fineness ratio, wings ~1.25-1.34 by thickness ratio, both
cruciform wing pairs counted), plus 15% for interference, plus a
bluff-body `frontal area x Cd` term for the open quad frame and exposed
rotor gear. Bigger wings genuinely pay for their extra skin, and the
teardrop earns its slender-body advantage from the same arithmetic that
charges the VTOL for its 2.6 m of wing.

In [ ]:
airframes = studies["ISR"].points["airframe"]
stories = {
    "boxQuad": "all bluff: open frame, battery, motor cans",
    "teardropQuad": "skinned lathe l/d 4.8 + bluff arms/motors",
    "vtolWing": "fuselage + BOTH wing pairs + 4 wingtip pods",
    "dartInterceptor": "slender body l/d 11 + thin wing + fins",
}
print(f"{'airframe':18s}{'CdA m^2':>9s}   drag story (the model's own buildup)")
for name, variant in airframes.variants.items():
    print(f"{name:18s}{variant['dragArea']:9.4f}   {stories[name]}")
print("\nTakeaway: the skin you fly is the drag you pay -- the dart's 11:1 body")
print("undercuts the teardrop 2:1, and both embarrass the open frame 4:1 and up.")

## Mission 1: ISR -- the wing buys the loiter

On station, the quad must *hover* (momentum-theory power from disk
loading) while the winged families fly slow on wing lift. Wing-borne
loiter costs less than a tenth of hover power, and the wingtip props'
induced-drag bonus stretches it further.

What to look for in the figure: the front is a real staircase. Budget
quads hold the cheap corner at 25-48 minutes, then the winged VTOL
takes over and runs past three hours -- the top step belongs to the
li-ion pack, whose extra watt-hours buy the last 47 minutes. The
staircase climbs in **material pairs**: at almost every step the
aluminum build is the cheaper seat and the carbon build buys a few
more minutes for a few more dollars. The interceptor is absent
entirely, because its 0.6 kg bay cannot carry the grade-2 gimbal the
mission requires. The pale crosses, some far
above the staircase, are *infeasible* mixes plotted at the metrics they
would score if they could fly: the constraints they break (`isrLift`,
`stationReq`, ...) are exactly why the front cannot reach them, so they
never join it.

In [ ]:
from longeron.analysis import viz

isr_front = trades.pareto(
    [a for a in spaces["ISR"] if a.verified],
    minimize=("missionCost",),
    maximize=("stationMinutes",),
)
isr_best = max(isr_front, key=lambda a: a.metrics["stationMinutes"])
isr_cheap = min(isr_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["ISR"],
    x="missionCost",
    y="stationMinutes",
    sense=("min", "max"),
    panel_y="missionMass",
    xlabel="mission cost (USD)",
    ylabel="time on station (min)",
    panel_ylabel="mission mass (kg)",
    annotate={"winged VTOL, li-ion pack: 209 min": isr_best, "budget quad corner": isr_cheap},
    title="The winged VTOL owns endurance; quads keep the cheap corner",
)

## Mission 2: logistics -- out heavy, back empty

The delivery radius is what the battery sustains for the *asymmetric*
round trip (outbound at parcel weight, return with the empty bay),
after a fixed hover budget for takeoff, drop-off, and landing.

What to look for in the figure: rotor-borne cruise never escapes hover
power, so the quad's radius stalls in the 8-16 km band, while the
winged VTOL turns the same packs into 30-163 kg-km of delivered
payload-range -- and the parcel lift wants LiPo watts, so the delivery
winner flies the 16 Ah Tattu, not the li-ion pack. Because every kilogram flies out AND back, the carbon
spar's saved grams show up here too. Neither the interceptor nor the
teardrop quad can even load the smallest parcel (`cargoFits`).

In [ ]:
log_front = trades.pareto(
    [a for a in spaces["logistics"] if a.verified],
    minimize=("missionCost",),
    maximize=("payloadRangeKgKm",),
)
log_best = max(log_front, key=lambda a: a.metrics["payloadRangeKgKm"])
log_cheap = min(log_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["logistics"],
    x="missionCost",
    y="payloadRangeKgKm",
    sense=("min", "max"),
    panel_y="deliveryRadiusKm",
    xlabel="mission cost (USD)",
    ylabel="payload x radius (kg km)",
    panel_ylabel="radius (km)",
    annotate={"winged VTOL: 4 kg out to 41 km": log_best, "$1236 quad: 1 kg to 8 km": log_cheap},
    title="Wings turn batteries into payload-range; quads just clear 8 km",
)

## Mission 3: intercept -- one-way dash, low drag wins

Dash speed is parasite-drag-limited (`(2 eta P / rho CdA)^(1/3)`), and
the reachable target speed inverts the lead-collision triangle at the
battery-limited dash duration. The catalog fields two dash
philosophies: the **winged dart** (a ~0.006 m^2 buildup, one 2 kW
AT4120 pusher) and the **wingless teardrop quad** (~0.0125 m^2, four
X4112S lifters).

What to look for in the figure: the dart owns the top of the front, the
teardrop takes a mid-price seat, and plain quads catch slow crossers
cheaply. The whole front is **LiPo and aluminum**: no li-ion pack can
feed the AT4120's draw, and dash physics never rewards the carbon
spar's grams. The few crosses above the staircase are infeasible sprint
mixes (four AT4120s out-draw every battery in the catalog).

In [ ]:
int_front = trades.pareto(
    [a for a in spaces["intercept"] if a.verified],
    minimize=("missionCost",),
    maximize=("maxTargetSpeed",),
)
int_best = max(int_front, key=lambda a: a.metrics["maxTargetSpeed"])
int_cheap = min(int_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["intercept"],
    x="missionCost",
    y="maxTargetSpeed",
    sense=("min", "max"),
    panel_y="dashSpeed",
    xlabel="mission cost (USD)",
    ylabel="max catchable target speed (m/s)",
    panel_ylabel="dash speed (m/s)",
    annotate={"2 kW dart: 73 m/s targets": int_best, "$1272 quad catches 28 m/s": int_cheap},
    title="The dart leads the dash; the wingless teardrop takes the mid-price front",
)

### Are wings necessary to intercept?

The teardrop quad exists to answer that question with physics instead
of opinion: same lift rotors as the box quad, a lathed teardrop shell
instead of an open frame, no wing. First-order truth: the dash cares
only about propulsive power over CdA, so a wing helps *only* insofar as
the whole airframe it belongs to is slicker. The drag buildup prices
the choice. The dart's 11:1 body plus thin wing reach ~0.006 m^2, but
its single motor caps power; the teardrop closes most of that gap with
four motors pushing a ~0.0125 m^2 shell. The model decides:

In [ ]:
dash_best = {}
for airframe in ("boxQuad", "teardropQuad", "dartInterceptor"):
    dash_best[airframe] = max(
        (a for a in spaces["intercept"] if a.verified and a.selection["airframe"] == airframe),
        key=lambda a: a.metrics["maxTargetSpeed"],
    )
gap = (
    dash_best["dartInterceptor"].metrics["maxTargetSpeed"]
    - dash_best["teardropQuad"].metrics["maxTargetSpeed"]
)
print(
    f"Wings buy {gap:+.1f} m/s at the top end -- necessary for the"
    if gap > 0
    else f"Wings cost {-gap:.1f} m/s at the top end -- the teardrop wins the",
    "fastest crossers, but the wingless teardrop is a genuine contender."
    if gap > 0
    else "dash outright.",
)
print()
print(f"{'best feasible dash mix':55s}{'dash':>7s}{'catch':>7s}{'cost':>7s}")
for airframe, arch in dash_best.items():
    mix = "/".join(arch.selection[k] for k in ("motors", "props", "battery", "material"))
    print(
        f"{airframe + '  ' + mix:55s}"
        f"{arch.metrics['dashSpeed']:7.1f}"
        f"{arch.metrics['maxTargetSpeed']:7.1f}"
        f"{arch.metrics['missionCost']:7.0f}"
    )

### Aluminum or carbon? The structure is sized, so the model can say

Rotor arms and wing spars are not fixed masses: the assembly sizes a
round tube for each, in the selected material, and the resulting mass
and cost feed every mission metric. The sizing uses bending stress with
a 2x safety factor AND a 2% tip-deflection stiffness floor; quad arms
are loaded by one rotor's max thrust at the prop-clearance arm length,
spars by the elliptic root bending moment at a 2.5 g load factor.
Carbon (1600 kg/m^3, ~110 GPa, $90/kg) buys grams over aluminum
(2700 kg/m^3, 69 GPa, $15/kg) at real money, so the choice lands
differently per mission:

In [ ]:
vtol_mix = {
    "airframe": "vtolWing",
    "motors": "x4112s",
    "props": "apc11x55",
    "battery": "tattu16000",
    "sensor": "zenmuseH20",
}
print(f"{'ISR winner, by spar material':34s}{'struct kg':>10s}{'cost $':>8s}{'station min':>12s}")
for material in ("aluminum", "carbonFiber"):
    arch = studies["ISR"].evaluate({**vtol_mix, "material": material})
    print(
        f"{material:34s}{arch.metrics['structureMass']:10.3f}"
        f"{arch.metrics['missionCost']:8.0f}{arch.metrics['stationMinutes']:12.1f}"
    )
front_mats = {
    name: sorted({a.selection["material"] for a in front})
    for name, front in (("ISR", isr_front), ("logistics", log_front), ("intercept", int_front))
}
print("\nmaterials holding front seats:", front_mats)
print("Carbon buys endurance and payload-range where mass is the currency;")
print("the dash never weighs its spar, so aluminum sweeps the intercept front.")

### Why the dead mixes die

Every infeasible mix names the constraints it breaks: the mix-level
answer CP-SAT's `explain()` gives at catalog level. The census per
mission is the design story in one table: sensors too heavy for small
bays, AT4120s out-drawing every pack, li-ion cells that cannot feed the
lifter motors, Antigravity motors that cannot lift the survey kit,
dashes that never catch the target.

In [ ]:
from collections import Counter

for name, archs in spaces.items():
    census = Counter(v for a in archs if not a.verified for v in a.violations)
    print(f"{name:9s}", dict(census.most_common()))

### The constraint network: who kills whom

The census counts corpses; the *structure* view shows the wiring.
`analysis.structure.constraint_network` draws the trade problem as a
bipartite graph: variation points (the decision variables) on the left,
`assert constraint` bodies on the right, and an edge wherever a
constraint's expression touches a point's selection *transitively
through the derived attributes*. `canCatch` compares `maxTargetSpeed`,
which is built from every component choice, so it wires to all four
points; `propFit` genuinely only couples props to motors.

What to look for in the widget: constraints that kill mixes in the
evaluated space are tinted warm with their body count. Hover either
side to light up its neighborhood.

In [ ]:
from longeron.analysis import structure

structure.constraint_network(studies["intercept"], spaces["intercept"])

## Across missions: is any one bird good at everything?

Project each mission's front onto the shared selection (airframe,
motors, props, battery, material). Several base mixes sit on *both* the
ISR and logistics fronts, above all the winged VTOL with standard
motors and slim props: buy that bird and re-fit the payload bay between
sorties. Nothing reaches all three fronts. The interceptor's dash
physics really is a different aircraft, and it shops in a different
aisle of the materials store.

In [ ]:
def base_mix(arch):
    keep = ("airframe", "motors", "props", "battery", "material")
    return tuple(arch.selection[k] for k in keep)


fronts = {"ISR": isr_front, "logistics": log_front, "intercept": int_front}
membership = {}
for name, front in fronts.items():
    for arch in front:
        membership.setdefault(base_mix(arch), set()).add(name)
print(f"{'airframe':16s}{'motors':13s}{'props':12s}{'battery':10s}{'material':13s} fronts")
for mix, names in sorted(membership.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    print(
        "".join(f"{part:13s}" if i else f"{part:16s}" for i, part in enumerate(mix))
        + "  "
        + ", ".join(sorted(names))
    )

### Brushing the mission space

`viz.parcoords` (the house anywidget: inline vanilla JS, brushes in a
narrow zone around each axis, editable intervals, `selected` synced
back to Python) carries one line per *base mix*, material included, so
the aluminum/carbon split is brushable. Each mix is scored on every
mission at once: each metric is the best that mix achieves over its
equipment options, 0 where no equipment choice is feasible.

What to look for in the widget: dashed gray lines fail every mission.
Brush `stationMinutes` high and `maxTargetSpeed` high to watch the
space empty out -- no line survives both.

In [ ]:
cross_rows = []
cross_mixes = []
for arch in spaces["intercept"]:  # the 288 shared base mixes
    mix = dict(
        zip(("airframe", "motors", "props", "battery", "material"), base_mix(arch), strict=True)
    )
    row = dict(mix)
    for name, metric in (
        ("ISR", "stationMinutes"),
        ("logistics", "payloadRangeKgKm"),
        ("intercept", "maxTargetSpeed"),
    ):
        best = [a for a in spaces[name] if a.verified and base_mix(a) == base_mix(arch)]
        row[metric] = max((a.metrics[metric] for a in best), default=0.0)
    row["cost"] = arch.metrics["missionCost"]
    row["feasible"] = any(
        row[m] > 0 for m in ("stationMinutes", "payloadRangeKgKm", "maxTargetSpeed")
    )
    cross_rows.append(row)
    cross_mixes.append(arch)

pc = viz.parcoords(
    cross_rows,
    axes=[
        "airframe",
        "motors",
        "props",
        "battery",
        "material",
        "cost",
        "stationMinutes",
        "payloadRangeKgKm",
        "maxTargetSpeed",
    ],
)
pc

## Shapes to scale: the 3D viewer

`analysis.geometry` bakes each family parametrically from the selected
catalog values (stdlib math, ~1 ms, no CAD kernel): the box quad's
frame from prop diameter + clearance; the teardrop quad's shell lathed
from a NACA-0025 thickness curve and stood on end, its long axis normal
to the planar rotor quad around it (in a dash the bullet flies
point-first); the winged VTOL as a cruciform **tail-sitter** (a minimal
slender fuselage, one long and one shorter unswept airfoil wing pair in
a `+`, and a rotor on each of the four wingtips with thrust parallel to
the chords/body axis), rendered nose-up in its hover attitude (it
pitches over to cruise wing-borne); and the interceptor as a lathed
slender body with a thin NACA-0009 wing and a pusher prop.

In the viewer below: drag to orbit, **shift-drag or right-drag to pan**
(the canvas swallows the context-menu event so JupyterLab stays out of
the way), scroll to zoom, and double-click to re-fit. The overlay hint
keeps the bindings visible. `viewer3d.mesh_viewer` fills the cell width
and re-fits on resize. (The viewer loads three.js from a CDN, the one
network dependency.)

In [ ]:
from longeron.analysis import geometry, viewer3d

viewer3d.mesh_viewer(
    geometry.mission_geometry(studies["ISR"], isr_best),
    label=(
        f"ISR winner in hover attitude -- {isr_best.metrics['stationMinutes']:.0f} min on station"
    ),
)

### Linked selection: parallel coordinates -> 3D

Plain traitlets compose the widgets: observe `selected` on the parallel
coordinates and re-bake the first surviving base mix into the viewer.
Brush the `airframe` axis through its categories and watch the shape
switch family.

In [ ]:
import json

linked = viewer3d.mesh_viewer(
    geometry.mission_geometry(studies["intercept"], cross_mixes[0]),
    label=" / ".join(base_mix(cross_mixes[0])),
)


def show_first_selected(change):
    indices = json.loads(change["new"] or "[]")
    if indices:
        mix = cross_mixes[indices[0]]
        linked.mesh_json = json.dumps(geometry.mission_geometry(studies["intercept"], mix))
        linked.label = " / ".join(base_mix(mix))


pc.observe(show_first_selected, names="selected")
linked

## The mission-compromise dashboard

The teaching steps above (parallel coordinates, linked 3D) compose into
one live artifact that ties **requirements -> architectures ->
performance -> cost**. `analysis.dashboard.mission_dashboard` wires four
panels together with plain traitlet observers (a couple hundred
candidates, so all logic stays in Python and every front-end stays a
dumb painter):

* **Mission requirements** -- one slider per threshold (ISR station
  floor, logistics payload + radius floors, intercept target speed),
  defaults read from the model's own requirement attributes. Every
  candidate's equipment options were baked with their achieved values,
  so moving a slider re-evaluates feasibility and re-picks each
  candidate's best equipment *live*. Tighten the ISR floor and watch
  quads drop out of the pool.
* **Mission priorities** -- weight sliders feeding the **MOE** (the
  documented compromise score: each mission's metric min-max normalized
  over its feasible set, weighted, `-0.5x` weight where a candidate
  cannot fly). The MOE is a **parallel-coordinates axis**, so brushing
  it works like any other column, and the whole table re-bakes as
  sliders move (brushes survive by axis name). A top-N slider sizes the
  3D lineup.
* The **MOE-vs-cost scatter** with the 2D Pareto front of the currently
  feasible candidates highlighted (`min` cost, `max` MOE) -- the
  buy-list view; picks wear a warm ring.
* Per-mission **requirement cards** (numeric margin per constraint,
  green holds / red broken; threshold rows track the sliders) and the
  **top-N 3D lineup** in an adaptive grid (4 -> 2x2, 6 -> 2x3), each
  cell captioned in-scene, the starred best compromise first.

Things to try: crank `intercept` to 100 and the dart takes the star;
hand the weight back to `ISR` and the tail-sitter returns; drag the
payload floor to 3 kg and logistics rebuilds around the large bay.
(Baking the candidate table walks 2016 interpreter-exact mixes.)

In [ ]:
from longeron.analysis import dashboard

dash = dashboard.mission_dashboard(model)
dash

## From feasible to preferred: the requirements scoreboard

The Pareto front narrowed 92 feasible mixes to the undominated few, but *undominated*
only means no mix beats it everywhere -- picking ONE still takes a statement
of what the stakeholder values. That statement belongs in the model, as a
requirement hierarchy: `weight` says what matters, `utility` says what good
looks like (`larger-is-better`, `smaller-is-better`, `target-is-best`, or a
pass/fail `step` backed by a real `require constraint`), and `measure` says
where the number comes from -- including derived expressions like
thrust-to-weight, evaluated by the interpreter against each architecture's
metrics. Tutorial 13 tours the machinery; here it picks the ISR aircraft.

In [ ]:
from longeron.analysis.scoreboard import architecture_values, scoreboard

isr_scoring = longeron.loads("""\
package IsrScoring {
    doc /* What the ISR stakeholder values in a mix, as a requirement
           hierarchy: weights say what matters, utility shapes say what
           good looks like, measures say where the number comes from. */

    requirement <'ISR-V'> isrValue {

        requirement <'ISR-1'> missionEffectiveness {
            doc /* Time over the target, unseen and unheard. */
            attribute weight : Real = 3.0;

            requirement <'ISR-11'> persistence {
                doc /* Minutes on station: barely useful at 30, saturating
                       at 2.5 hours. */
                attribute weight : Real = 2.0;
                attribute utility : String = "larger-is-better";
                attribute ramp0 : Real = 30.0;
                attribute ramp1 : Real = 150.0;
                attribute measure : Real = stationMinutes;
            }
            requirement <'ISR-12'> covertness {
                doc /* Loiter power is the acoustic-signature proxy:
                       600 W is conspicuous, 100 W is near-silent. */
                attribute weight : Real = 1.0;
                attribute utility : String = "smaller-is-better";
                attribute ramp0 : Real = 600.0;
                attribute ramp1 : Real = 100.0;
                attribute measure : Real = loiterPowerW;
            }
        }

        requirement <'ISR-2'> affordability {
            attribute weight : Real = 2.0;

            requirement <'ISR-21'> unitCost {
                doc /* Full mission kit against the program's budget --
                       the H20 gimbal alone is $4,600 of it. */
                attribute weight : Real = 3.0;
                attribute utility : String = "smaller-is-better";
                attribute ramp0 : Real = 8000.0;
                attribute ramp1 : Real = 5500.0;
                attribute measure : Real = missionCost;
            }
            requirement <'ISR-22'> repairability {
                doc /* Structure cost proxies field-repair pain: carbon
                       spars are light but nobody splints one in a tent. */
                attribute weight : Real = 1.0;
                attribute utility : String = "smaller-is-better";
                attribute ramp0 : Real = 15.0;
                attribute ramp1 : Real = 1.0;
                attribute measure : Real = structureCost;
            }
        }

        requirement <'ISR-3'> fieldability {
            doc /* Carried by one soldier, launched from a clearing. */
            attribute weight : Real = 2.0;

            requirement <'ISR-31'> portability {
                doc /* 3.5 kg rides in one rucksack; 7 kg takes two. */
                attribute weight : Real = 2.0;
                attribute utility : String = "smaller-is-better";
                attribute ramp0 : Real = 7.0;
                attribute ramp1 : Real = 3.5;
                attribute measure : Real = missionMass;
            }
            requirement <'ISR-32'> packability {
                doc /* Arm length drives the folded footprint. */
                attribute weight : Real = 1.0;
                attribute utility : String = "smaller-is-better";
                attribute ramp0 : Real = 0.40;
                attribute ramp1 : Real = 0.15;
                attribute measure : Real = armLength;
            }
        }

        requirement <'ISR-4'> flightRobustness {
            attribute weight : Real = 1.0;

            requirement <'ISR-41'> hoverAuthority {
                doc /* Thrust-to-weight: 2.0 holds station in gusts; much
                       more is dead motor mass, much less is a brick. */
                attribute utility : String = "target-is-best";
                attribute target : Real = 2.0;
                attribute limit : Real = 1.0;
                attribute measure : Real = maxThrust / (9.81 * missionMass);
            }
            requirement <'ISR-42'> energyReserve {
                doc /* Hard floor: land with a real reserve, not fumes. */
                attribute utility : String = "step";
                require constraint { usableEnergyJ >= 400.0e3 }
            }
        }
    }
}\n""")

isr_feasible = [a for a in spaces["ISR"] if a.verified]
isr_ranked = sorted(
    isr_feasible,
    key=lambda a: scoreboard(isr_scoring, values=architecture_values(a)).score,
    reverse=True,
)
for arch in isr_ranked[:3]:
    sb = scoreboard(isr_scoring, values=architecture_values(arch))
    print(
        f"{sb.score:.3f}  station={arch.metrics['stationMinutes']:6.1f}  "
        f"cost={arch.metrics['missionCost']:7.0f}  {arch.selection}"
    )
isr_preferred = isr_ranked[0]
print()
print(scoreboard(isr_scoring, values=architecture_values(isr_preferred)))

The winner flies the Antigravity motors on the 16 Ah Tattu with an aluminum
spar -- and beats the li-ion endurance king by a whisker, giving back 47
minutes of station time to score better on cost and repairability. The
table already names the soft spot (portability: 5.4 kg is a two-person
carry), and weakest-link aggregation (`aggregation="min"`) makes that the
headline instead of averaging it away. The same decomposition, drawn: area
is weight, color is utility, hatching is unmeasured; double-click a group
cell to zoom into it, and the ▸/▾ twist collapses a group to its
aggregate in place.

In [ ]:
scoreboard(isr_scoring, values=architecture_values(isr_preferred)).widget(tessellation="voronoi")

## Continuous sizing: how fast should the ISR winner loiter?

Trades picked *which* components; `analysis.mdao` sizes what stays
continuous. `UavMissions::IsrPrime` freezes the ISR winner's component
mix as a concrete part definition and leaves `loiterSpeed` free. It
spells its physics out **discipline by discipline** with the shared
calc defs: Aerodynamics builds the drag, Structures sizes the spar and
its mass, Propulsion turns mass and drag into power and usable energy,
Performance turns power into minutes on station. `build_problem`
mirrors the part onto an OpenMDAO `Problem` (attributes become
components evaluating through the interpreter; constraints and the
`IsrStation` requirement become `*_margin` outputs) and **groups the
components by the calc defs' owning packages** -- the model's structure
is the problem's structure.

In [ ]:
build = mdao.build_problem(
    model, "UavMissions::IsrPrime", requirements=("UavMissions::IsrStation",)
)
p = build.problem
p.run_model()
for discipline, attrs in build.disciplines.items():
    print(f"{discipline:14s} {', '.join(attrs)}")
print("loiterPowerW:  ", round(float(p.get_val("loiterPowerW")[0]), 1))
print("stationMinutes:", round(float(p.get_val("stationMinutes")[0]), 1))
p.set_val("loiterSpeed", 21.0)  # what-if: loiter at transit speed
p.run_model()
print(
    "at 21 m/s:     ",
    round(float(p.get_val("stationMinutes")[0]), 1),
    "min -- stationFloor margin",
    round(float(p.get_val("stationFloor_margin")[0]), 1),
)
p.set_val("loiterSpeed", 15.0)
p.run_model()

### The problem's shape: an N2 map

Before trusting numbers from a generated `Problem`, look at its
structure. `analysis.structure.n2_view` introspects the built problem's
components and global connections and draws the classic N2 matrix in
the NASA/OpenMDAO convention: components on the diagonal in execution
order, each coupling in its **source's row and target's column**, so
the flow reads clockwise and **feed-forward fills the upper triangle**.

What to look for in the widget: the dashed outlines are the
**discipline blocks** (`Aerodynamics`, `Structures`, `Propulsion`,
`Performance`), taken from the OpenMDAO groups `build_problem` derived
from the model's own calc packages. Drag flows out of Aerodynamics into
Propulsion, the sized spar mass into the mass roll-up, and power into
Performance -- exactly the story an MDO engineer expects the matrix to
tell. Hover a dot to list the coupled variables; click to pin the
tooltip. This sizing chain is a pure feed-forward cascade, so every dot
sits above the diagonal; a feedback coupling would land *below* it,
warm and dash-ringed.

In [ ]:
structure.n2_view(build)

### The same problem in OpenMDAO's own N2

For the full-strength deep dive, `structure.openmdao_n2` embeds the
official diagram (`openmdao.api.n2`, generated headlessly, returned as
a self-contained iframe): the identical clockwise convention plus
everything the house widget deliberately leaves out. Because the
disciplines are real OpenMDAO groups, the official hierarchy shows
`Aerodynamics`, `Structures`, `Propulsion`, and `Performance` as
collapsible blocks too. Use the house `n2_view` for a quick
dependency-free read; open the official one when you need to
interrogate a big model.

In [ ]:
structure.openmdao_n2(build)

### The margin picture

Sweeping `loiterSpeed` *past* the legal window shows every wall at
once, and **only the walls are shaded**: wherever any margin goes
negative the band is hatched warm and labeled with every constraint
binding there (stacked when several overlap). The unshaded middle *is*
the feasible corridor.

What to look for in the figure: below ~11 m/s the `aboveStall` floor
binds; past ~19.5 m/s the 90-minute `stationFloor` gives out; and
beyond 24 m/s the `belowCruise` ceiling piles on top of the
already-broken station floor. One chart, three different reasons the
design cannot go there.

In [ ]:
fig = viz.margin_sweep_figure(
    p,
    "loiterSpeed",
    [9.0 + 0.35 * i for i in range(50)],
    build.constraints,
    xlabel="loiter speed (m/s)",
    title="The station floor caps loiter at ~19.5 m/s; stall and transit limits frame the corridor",
)

Maximizing `stationMinutes` with SLSQP drives the loiter speed straight
into the stall bound: by the first-order drag polar, the slower the
better.

In [ ]:
opt = mdao.build_problem(
    model, "UavMissions::IsrPrime", setup=False, requirements=("UavMissions::IsrStation",)
)
mdao.add_optimization(
    opt, objective="stationMinutes", design_vars={"loiterSpeed": (11.0, 24.0)}, maximize=True
)
opt.problem.setup()
opt.problem.set_val("loiterSpeed", 16.0)
opt.problem.run_driver()
print(
    f"best loiter = {opt.problem.get_val('loiterSpeed')[0]:.2f} m/s "
    f"({opt.problem.get_val('stationMinutes')[0]:.0f} min on station)"
)

## Declared external analyses: swapping the aerodynamics fidelity

First-order physics belongs in the model as `calc def` bodies -- but
higher-fidelity tools live outside SysML. The convention shipped with the
example makes the *model* declare the binding:

```sysml
metadata def ExternalAnalysis { attribute component : String; }

calc def CruisePower {
    @ExternalAnalysis { component = "uav_aero:CruisePowerPolar"; }
    in massKg : Real;  in speed : Real;  ...
    return : Real = ...first-order drag polar...;
}
```

The calc's `in`/`return` parameters *are* the I/O contract.
`build_problem` validates them against the wrapped OpenMDAO component's
actual inputs/outputs (a mismatch fails with both name lists) and
`fidelity={"CruisePower": "external"}` swaps the interpreter-backed body
for the component -- here `examples/uav_aero.py`, a synthetic
Reynolds-and-stall-aware polar. Everything else in the Problem is
untouched, so lo-fi/hi-fi comparison is one keyword:

In [ ]:
import sys

if "../examples" not in sys.path:
    sys.path.insert(0, "../examples")  # the uav_aero entry point

lo = mdao.build_problem(model, "UavMissions::IsrPrime")
hi = mdao.build_problem(model, "UavMissions::IsrPrime", fidelity={"CruisePower": "external"})
print("bound externals:", hi.externals)

speeds = [11.0 + 0.2 * i for i in range(51)]
station = {}
for name, b in (("first-order calc body", lo), ("uav_aero polar (external)", hi)):
    values = []
    for v in speeds:
        b.problem.set_val("loiterSpeed", v)
        b.problem.run_model()
        values.append(float(b.problem.get_val("stationMinutes")[0]))
    station[name] = values

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.0, 3.6), layout="constrained")
for (name, values), color in zip(station.items(), ("#2f6b8f", "#c2603e"), strict=True):
    ax.plot(speeds, values, color=color, linewidth=1.6)
    best = max(range(len(speeds)), key=lambda i: values[i])
    ax.plot(speeds[best], values[best], "o", color=color, markersize=5)
    ax.annotate(
        f"{name}\nbest {values[best]:.0f} min @ {speeds[best]:.1f} m/s",
        (speeds[best], values[best]),
        xytext=(10, -6),
        textcoords="offset points",
        fontsize=8,
        color=color,
    )
ax.set_xlabel("loiter speed (m/s)")
ax.set_ylabel("time on station (min)")
ax.set_title(
    "The Reynolds/stall-aware polar backs loiter off the stall and costs 40 min",
    fontsize=10,
    loc="left",
    color="#2b2d31",
)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#d9dbdf", linewidth=0.5)

The first-order body rewards flying ever slower; the external polar's
stall-adjacent drag rise pushes the optimum to ~12 m/s and takes the
promised endurance from 300 to 246 minutes. Same model, same declared
contract, one keyword -- that is what the annotation is for. (The
convention is a candidate for a shared library package once it has
earned its keep here.)

## Requirement consistency with Z3

`analysis.smt` answers questions over *unbounded reals*: is the
requirement set satisfiable at all, and what exactly bounds it? The
`IsrStation` requirement (90 minutes on station) is consistent with
`IsrPrime`'s physics. Freeing `loiterSpeed` shows the *exact* fastest
loiter that still satisfies it:

In [ ]:
system = smt.to_smt(model, "UavMissions::IsrPrime", requirements=("UavMissions::IsrStation",))
result = system.check()
print(
    result.status,
    {k: round(v, 1) for k, v in result.witness.items() if k in ("loiterSpeed", "stationMinutes")},
)

freed = smt.to_smt(
    model, "UavMissions::IsrPrime", requirements=("UavMissions::IsrStation",), free=("loiterSpeed",)
)
bound, _ = freed.maximize("loiterSpeed")
print("fastest loiter satisfying the 90 min floor:", bound, "m/s")

Now demand 360 minutes -- more than the aircraft has at any legal speed --
via programmatic authoring (notebook 1), and ask *which* requirements
collide. The core names `DeepStare::longStation`
against the legal loiter-speed window (`aboveStall`/`belowCruise` --
only speeds outside it could stretch the battery that far) through the
defining equations, and correctly excludes the satisfiable 90-minute
floor:

In [ ]:
from longeron import model as M

deep = M.Definition(kind="requirement", name="DeepStare")
deep.add(M.Usage(kind="subject", name="uav", types=["IsrPrime"]))
deep.add(
    M.Usage(
        kind="constraint",
        name="longStation",
        constraint_kind="require",
        result=longeron.parse_expression("uav.stationMinutes >= 360.0"),
    )
)
model.find("UavMissions").add(deep)

conflicted = smt.to_smt(
    model,
    "UavMissions::IsrPrime",
    requirements=("UavMissions::IsrStation", "UavMissions::DeepStare"),
    free=("loiterSpeed",),
)
result = conflicted.check()
print(result.status)
for label in result.core:
    if not label.endswith(".value"):
        print(" ", label)

## Where this goes: the two-level loop

The pieces compose into classic mixed-discrete MDO: **the trade study
picks the architecture, OpenMDAO sizes it**. `all_architectures()`
scores every mix exactly through the interpreter; the winner freezes
into a concrete sizing context whose continuous attributes SLSQP
optimizes (swapping declared external analyses in for the physics that
outgrows first-order calc bodies), while Z3 guards the requirement set
before any solver time is spent on an impossible ask.

### Real CAD when you need it

For actual CAD output -- STEP for a printable frame --
`geometry.to_cadquery` rebuilds the quad parametrically as cadquery
solids behind the `cad` extra (the OCC kernel is ~1 GB, which is why
the mesh pipeline above never touches it). This cell degrades to a note
when cadquery is not installed:

In [ ]:
log_quad = min(
    (a for a in spaces["logistics"] if a.verified and a.selection["airframe"] == "boxQuad"),
    key=lambda a: a.metrics["missionCost"],
)
try:
    params = geometry.mission_params(studies["logistics"], log_quad)
    assembly = geometry.to_cadquery(
        prop_diameter_in=params["prop_diameter"] / geometry.IN,
        motor_mass=params["motor_mass"],
        battery_mass=params["battery_mass"],
        esc_mass=0.014,
    )
    print(
        f"cadquery assembly with {len(assembly.children)} parts -- "
        "assembly.export('drone.step') exports STEP"
    )
except ImportError:
    print('optional: pip install "longeron[cad]" enables STEP export -- skipping here')

## Find my violations: `longeron.analysis.verify`

Everything above shows the model *answering questions*;
`verify` makes it fight back. Four tiers over one oracle -- every
verdict comes from the interpreter, solvers only *propose* -- all
derived from nothing but the `.sysml` text: `hunt` (Hypothesis
sampling + shrinking over strategies mined from attribute types,
`assert` bodies, and Z3 bounds through the derivation chain),
`sequences` (adversarial events against the real state machine),
`cover` (in-house IPOG t-way covering arrays), and `prove` (Z3
absence proofs with exact rational bounds). The sampling tiers need
the `verify` extra: `pip install "longeron[verify]"`.

In [ ]:
import importlib.util

from longeron.analysis import verify

HAS_HYPOTHESIS = importlib.util.find_spec("hypothesis") is not None
if not HAS_HYPOTHESIS:
    print(
        "hunt/sequences skipped below -- Hypothesis is not installed. "
        'Run: pip install "longeron[verify]"'
    )

### Hunt: the shrunk catch, then the exact edges

Free `payloadMass` on the drone and the domain ladder goes to work:
no constraint bounds it *directly*, but Z3, driven through the same
`totalMass` derivation chain the interpreter evaluates, extracts the
exact window the `FlightEnvelope` assumption implies. Hypothesis then
shrinks to the *simplest* violator -- deliberately not the tightest --
and the report pairs it with interpreter-bisected edges per violated
check, so nobody has to choose between a debuggable repro and the
true boundary.

In [ ]:
drone = longeron.load("../examples/drone.sysml")
if HAS_HYPOTHESIS:
    hunt_report = verify.hunt(
        drone,
        "Drone::QuadCopter",
        requirements=("Drone::FlightEnvelope",),
        free=("payloadMass",),
        seed=0,
    )
    dom = hunt_report.domains["payloadMass"]
    hi = dom.hi if dom.hi is not None else "unbounded (fallback, flagged)"
    print(f"derived window : payloadMass in [{dom.lo}, {hi}] [{dom.unit}]")
    catch = hunt_report.counterexamples[0]
    print(f"shrunk catch   : {catch.bindings}  violates {list(catch.violated)}")
    for edge in sorted(hunt_report.boundaries, key=lambda b: b.value):
        print(f"exact edge     : payloadMass = {edge.value:.6f} kg flips {edge.violated}")
else:
    print('skipped: pip install "longeron[verify]"')

### Sequences: the minimal violating sortie

`Drone::SortieStates` guards *launches* behind a 30% battery floor,
but its go-around path re-enters `airborne` without repassing the
guard. One generic rule (send any event from the alphabet read off
the transitions) plus one invariant (`SafeSortie` against the live
simulation environment) finds the trap and shrinks away every
irrelevant event.

In [ ]:
import sys

sys.executable

In [ ]:
if HAS_HYPOTHESIS:
    seq_report = verify.sequences(
        drone, "Drone::SortieStates", requirements=("Drone::SafeSortie",), seed=0
    )
    sortie = seq_report.counterexamples[0]
    print("minimal sortie :", " -> ".join(sortie.events))
    print("violates       :", list(sortie.violated))
else:
    print('skipped: pip install "longeron[verify]"')

### Cover: t-way arrays over the ISR catalog, recall *measured*

The same variation points the trade study enumerated become covering-
array factors (in-house IPOG, no new dependency); every row is
settled interpreter-exact. At catalog scale the exhaustive space is
still enumerable, so the recall line below is a measurement against
ground truth, not a guarantee -- pairwise promises pair coverage,
and the report says exactly that when exhaustion stops being
feasible.

In [ ]:
cover_report = verify.cover(model, "UavMissions::IsrUav", t=2)
cov = cover_report.coverage
print(f"array          : {len(cov.rows)} pairwise rows vs {cov.exhaustive} exhaustive mixes")
print(f"violations     : {cover_report.violations}")
print(f"measured recall: {cov.recall:.0%} of the checks any exhaustive mix violates")

### Prove: violation is *impossible* (and the exact ceiling)

Negating one check at a time under the assumptions and the other
checks: inside the takeoff-mass budget, a `hoverMargin` violation is
UNSAT -- a proof of absence no amount of sampling delivers -- while
the budget itself *can* be busted (witness re-checked by the
interpreter before it is believed). `Optimize` hands back the exact
rational payload ceiling, attributed to the constraint that binds it.

In [ ]:
prove_report = verify.prove(
    drone,
    "Drone::QuadCopter",
    requirements=("Drone::FlightEnvelope",),
    free=("payloadMass",),
)
for proof in prove_report.proofs:
    print(f"{proof.requirement:42s} {proof.status}")
top = prove_report.proofs[0]
print(f"\nmax feasible payload = {top.bound} kg exactly, bound by {top.binding_constraint}")

Every catch closes the loop into M0: `materialize()` turns the
shrunk bindings into identified individuals, re-checkable with the
ordinary machinery (and `verify.counterexample_values` feeds them to
the scoreboard as `values=` bindings -- the red-cell moment).

In [ ]:
if HAS_HYPOTHESIS:
    individual = catch.materialize()
    print(individual.root.id, ": totalMass =", individual.root.slots["totalMass"], "kg")
    for check in longeron.Interpreter(drone).check(individual.root):
        print(f"  {check.name:18s} passed={check.passed}")
else:
    print('skipped: pip install "longeron[verify]"')

## Objects across the bridge: entities and file artifacts

Scalars are not the only things that cross into OpenMDAO
([design](../design/mdao-objects.md)). Passing an **M0
interpretation** to `build_problem` turns each variation point into a
*discrete* input carrying the configured individual -- the ISR winner's
mix builds directly on `IsrUav`, no hand-frozen `IsrPrime` needed --
and `bind_entity` swaps a case in place: hand it a different motor and
the whole sizing chain re-evaluates. `record_case` then freezes the
evaluated case as an immutable interpretation snapshot: outputs land as
attribute values on the case's individuals (stable ids, roll-ups over
the recorded population).

In [ ]:
from longeron import m0

case = m0.interpret(model, "UavMissions::IsrUav", selection=isr_preferred.selection)
entity_build = mdao.build_problem(model, "UavMissions::IsrUav", interpretation=case)
ep = entity_build.problem
ep.run_model()
print("entity inputs:", ", ".join(sorted(entity_build.entities)))
print(
    f"stationMinutes ({case.selection['motors']}):",
    round(float(ep.get_val("stationMinutes")[0]), 1),
    "min",
)
mdao.bind_entity(entity_build, "motors", "UavMissions::SunnySkyX4112s")
ep.run_model()
print("stationMinutes (x4112s):   ", round(float(ep.get_val("stationMinutes")[0]), 1), "min")
snapshot = mdao.record_case(entity_build)
print(
    "recorded case:",
    snapshot.root.id,
    "| motors =",
    snapshot.selection["motors"],
    "| motor mass roll-up:",
    round(snapshot.rollup("sum(motors.mass)"), 3),
    "kg",
)

Files cross the same way: a `FileArtifact` -- path plus sha256 content
identity -- flows between components as ~200 bytes of discrete value
while the bytes stay on disk (`ExternalCodeComp`-compatible, and the
hash is the caching identity: same recipe, same hash, skip the external
run). The payload stays **M0-keyed**: the individual id says exactly
which configured part the geometry belongs to.